# Decision-making under ambiguity and poverty (DMAP)

A notebook that gives an overview and does a test run of the agent-based model for decision-making under ambiguity and poverty (DMAP).

`theta` = wealth

`gamma` = utility parameter

`beta` = optimism/pessimism

`alpha` = likelihood sensitvity

`p` = probability of being caught

# TO DO:

- See how to store decisions
- Figure out how to update wealth after decision
- ~~See how to strcture decision_maker class and whether it requires more than just __init__ and step()~~

# agents.py

Constructing individual decision-maker and the rules they follow

In [2]:
import mesa
import numpy as np
from scipy.stats import bernoulli
import matplotlib.pyplot as plt
import seaborn as sns
import cognitive_functions as cf


In [3]:
class decision_maker(mesa.Agent):
    """An agent that chooses between two options based on expected utility."""
    
    def __init__(self, unique_id, model, gamma, reward_rb, reward_rf, cost_rb, wealth, p, beta, alpha):
        """ Create a new decision maker agent.
        Args:
            unique_id: Unique identifier for the agent.
            model: Reference to the model this agent belongs to.
            gamma: Utility parameter.
            reward_rb: Reward for rule breaking.
            reward_rf: Reward for following rules.
            cost_rb: Cost of rule breaking.
            wealth: Wealth of the agent.
            p: Probability of being caught for rule-breaking.
            beta: Optimism/pessimism.
            alpha: Likelihood sensitivity.
        """
        super().__init__(unique_id, model)

    def compute_expected_utilities(self, gamma, reward_rb, reward_rf, cost_rb, p, beta, alpha):
        self.EU_rule_break = cf.EU_rule_break(gamma, reward_rb, reward_rf, cost_rb, p, beta, alpha)  # EU of rule breaking
        self.EU_follow_rules = cf.EU_follow_rules(gamma, reward_rb, reward_rf, cost_rb, p, beta, alpha) # EU of following rules

    def step(self):
        """Choose an option based on expected utility."""
        
        EUs = np.array([self.EU_rule_break, self.EU_follow_rules])
        probs = cf.softmax(EUs)
        
        choice = bernoulli.rvs(probs[0])
        if choice:
            print(f"Agent {self.unique_id} chose to break the rules with probability {probs[0]}")
        else:
            print(f"Agent {self.unique_id} chose to follow the rules with probability {probs[1]}")
        return choice

# model.py

Creating model environment to run the ABM.

In [ ]:
from mesa import Model
from mesa.datacollection import DataCollector
import numpy as np

# from .agents import decision_maker

class DMAP_model(Model):
    """A model with a number of decision makers."""
    
    def __init__(self, N, gamma, reward_rb, reward_rf, cost_rb, wealth, p, beta, alpha):
        super().__init__(N, gamma, reward_rb, reward_rf, cost_rb, wealth, p, beta, alpha)
        self.num_agents = N
        self.gamma = gamma
        self.reward_rb = reward_rb
        self.reward_rf = reward_rf
        self.cost_rb = cost_rb
        self.wealth = wealth
        self.p = p
        self.beta = beta
        self.alpha = alpha
        
        # Create agents
        self.schedule = mesa.time.RandomActivation(self)
        for i in range(self.num_agents):
            agent = decision_maker(i, self, gamma, reward_rb, reward_rf, cost_rb, wealth, p, beta, alpha)
            self.schedule.add(agent)
        
        # Data collector
        self.datacollector = DataCollector(
            agent_reporters={"Choice": "step"}
        )

## Run the model

In [ ]:
starter_model = DMAP_model(10)
starter_model.step()

In [ ]:
model = MoneyModel(10)  # Tells the model to create 10 agents


In [ ]:

for _ in DMAP_model(30):  # Runs the model for 30 steps;
    model.step()

# Note: An underscore is common convention for a variable that is not used.

In [ ]:
agent_wealth = [a.wealth for a in model.agents]
# Create a histogram with seaborn
g = sns.histplot(agent_wealth, discrete=True)
g.set(
    title="Wealth distribution", xlabel="Wealth", ylabel="number of agents"
);  # The semicolon is just to avoid printing the object representation